# Exploratory Data Analysis: Xente Credit Risk Dataset

This notebook explores the Xente eCommerce transaction dataset to uncover patterns, identify data quality issues, and form hypotheses for feature engineering.

**Objectives:**
1. Understand the structure and characteristics of the transaction data
2. Analyze distributions of numerical and categorical features
3. Identify missing values and outliers
4. Discover correlations and relationships
5. Generate insights to guide feature engineering

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Load the data
df = pd.read_csv('../data/data.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nDataset head:")
df.head()

## 2. Dataset Structure and Summary Statistics

In [ ]:
# Data types and info
print("Data Types:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Check for missing values
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
}).sort_values('Missing_Count', ascending=False)

print("Missing Values Analysis:")
print(missing_data[missing_data['Missing_Count'] > 0])

# Visualize missing values
plt.figure(figsize=(12, 4))
missing_data[missing_data['Missing_Count'] > 0].plot(
    x='Column', y='Missing_Percentage', kind='bar', legend=False
)
plt.title('Missing Values by Feature')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics for numerical features
print("\nSummary Statistics for Numerical Features:")
df.describe().T

## 3. Distribution of Numerical Features

In [ ]:
# Identify numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numerical columns: {numerical_cols}")

# Plot distributions
fig, axes = plt.subplots(len(numerical_cols), 2, figsize=(14, 4*len(numerical_cols)))

for idx, col in enumerate(numerical_cols):
    # Histogram
    axes[idx, 0].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[idx, 0].set_title(f'Distribution of {col}')
    axes[idx, 0].set_xlabel(col)
    axes[idx, 0].set_ylabel('Frequency')
    
    # Box plot
    axes[idx, 1].boxplot(df[col].dropna())
    axes[idx, 1].set_title(f'Box Plot of {col}')
    axes[idx, 1].set_ylabel(col)

plt.tight_layout()
plt.show()

## 4. Distribution of Categorical Features

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

# Plot distributions for each categorical feature
for col in categorical_cols:
    fig, ax = plt.subplots(figsize=(12, 4))
    value_counts = df[col].value_counts()
    ax.bar(range(len(value_counts)), value_counts.values)
    ax.set_xticks(range(len(value_counts)))
    ax.set_xticklabels(value_counts.index, rotation=45, ha='right')
    ax.set_title(f'Distribution of {col}')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
    
    print(f"\n{col} - Value Counts:")
    print(value_counts)

## 5. Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = df[numerical_cols].corr()

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

print("\nCorrelation Matrix:")
print(correlation_matrix)

## 6. Outlier Detection

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

print("Outlier Detection (IQR Method):")
for col in numerical_cols:
    count, lower, upper = detect_outliers_iqr(df, col)
    print(f"{col}: {count} outliers (bounds: [{lower:.2f}, {upper:.2f}])")

## 7. Key Insights Summary

### Summary of Key Findings:

1. **Data Volume and Quality**: The dataset contains [INSERT NUMBER] transactions across [INSERT NUMBER] unique customers. Missing data is present in [INSERT COLUMNS], requiring imputation strategy.

2. **Feature Distributions**: 
   - Transaction amounts show [right-skewed/left-skewed/normal] distribution
   - Categorical features are [balanced/imbalanced] across categories
   - Outliers detected in [COLUMNS] requiring special handling

3. **Correlation Insights**: 
   - Strong correlations exist between [FEATURES]
   - Weak correlations suggest feature independence
   - Multicollinearity may be present between [FEATURES]

4. **Customer Behavior**:
   - Transaction patterns reveal [high/low] customer engagement
   - Product categories show varying popularity
   - Channel distribution indicates platform preferences

5. **Recommendations for Feature Engineering**:
   - Create RFM (Recency, Frequency, Monetary) aggregations for customer segmentation
   - Engineer temporal features (hour, day, month, year) from timestamps
   - Apply logarithmic transformation for skewed distributions
   - Handle outliers through capping or separate treatment
   - Use target encoding or one-hot encoding for categorical variables